# 104 - Greeks and Volatility Surface

Three-dimensional visualization of the option Greeks for SPY. We pull
Greeks straight from the snapshot endpoint - the SDK computes the full
Black-Scholes set server-side and returns them per contract, so each row
already carries the implied volatility, delta, gamma, theta, vega, and the
higher-order Greeks.

1. Fetch several expirations spanning near-term to a few months out
2. Pull snapshot Greeks across strikes for each expiration
3. Build the IV surface (moneyness x expiration x IV)
4. 3D surface plot of implied volatility
5. Delta / gamma / theta heatmaps across the surface
6. Time-decay analysis: ATM Greeks as expiration approaches

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from datetime import datetime

from thetadatadx import Credentials, Config, ThetaDataDxClient

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

In [ ]:
creds = Credentials.from_file("creds.txt")
tdx = ThetaDataDxClient(creds, Config.production())

## 1. Fetch Multiple Expirations

Pick expirations near 7, 14, 30, 60, 90, and 120 days out. Expirations
are ISO date strings.

In [ ]:
all_exps = tdx.option_list_expirations("SPY").to_list()
today = datetime.now()

target_dtes = [7, 14, 30, 60, 90, 120]
selected_exps = []

for target in target_dtes:
    best, best_diff = None, 10**9
    for exp_str in all_exps:
        exp_date = datetime.strptime(exp_str, "%Y-%m-%d")
        dte = (exp_date - today).days
        if dte > 0 and abs(dte - target) < best_diff:
            best_diff, best = abs(dte - target), exp_str
    if best and best not in selected_exps:
        selected_exps.append(best)

for exp_str in selected_exps:
    dte = (datetime.strptime(exp_str, "%Y-%m-%d") - today).days
    print(f"  {exp_str}  ({dte} DTE)")

In [ ]:
# Spot from a stock OHLC snapshot (one row per symbol).
spot = tdx.stock_snapshot_ohlc(["SPY"])[0].close
print(f"SPY spot: ${spot:.2f}")

## 2. Pull Snapshot Greeks Across Strikes

For each expiration, fetch the snapshot Greeks for the calls within +/-8%
of spot. `option_snapshot_greeks_all` is keyword-only for `expiration` /
`strike` / `right`, takes the strike as a dollar string, and returns a
list of typed rows carrying `implied_volatility`, `delta`, `gamma`,
`theta`, `vega`, and more.

In [ ]:
surface_data = []

for exp_str in selected_exps:
    dte = (datetime.strptime(exp_str, "%Y-%m-%d") - today).days

    # Strikes within +/-8% of spot, as dollar strings.
    strikes_raw = tdx.option_list_strikes("SPY", exp_str).to_list()
    strikes_f = [(s, float(s)) for s in strikes_raw]
    strikes_f = [(s, v) for s, v in strikes_f if spot * 0.92 <= v <= spot * 1.08]

    print(f"  {exp_str} ({dte:3d} DTE): {len(strikes_f)} strikes", end="")

    count = 0
    for strike_str, strike_val in strikes_f:
        try:
            rows = tdx.option_snapshot_greeks_all(
                "SPY", expiration=exp_str, strike=strike_str, right="C")
        except Exception:
            continue
        if not rows:
            continue
        g = rows[0]
        if g.implied_volatility is None or g.implied_volatility <= 0:
            continue
        surface_data.append({
            "expiration": exp_str,
            "dte": dte,
            "strike": strike_val,
            "moneyness": strike_val / spot,
            "iv": g.implied_volatility,
            "delta": g.delta,
            "gamma": g.gamma,
            "theta": g.theta,
            "vega": g.vega,
            "vanna": g.vanna,
            "charm": g.charm,
            "vomma": g.vomma,
        })
        count += 1

    print(f" -> {count} valid")

surface = pd.DataFrame(surface_data)
print(f"\nTotal data points: {len(surface)}")

## 3. Build the Volatility Surface

Pivot to a grid: moneyness on one axis, days-to-expiration on the other,
implied volatility as the value. Moneyness is bucketed to 0.5% steps for a
cleaner grid.

In [ ]:
surface["moneyness_bucket"] = (surface["moneyness"] * 200).round() / 200

iv_pivot = surface.pivot_table(
    index="moneyness_bucket", columns="dte", values="iv", aggfunc="mean"
)

print(f"Surface grid: {iv_pivot.shape[0]} moneyness levels "
      f"x {iv_pivot.shape[1]} expirations")
iv_pivot.head()

## 4. Plot the IV Surface

In [ ]:
fig = plt.figure(figsize=(14, 8))
ax = fig.add_subplot(111, projection="3d")

X = iv_pivot.columns.values   # DTE
Y = iv_pivot.index.values     # Moneyness
X_grid, Y_grid = np.meshgrid(X, Y)
Z = iv_pivot.values * 100     # IV in %

# Interpolate gaps for a smoother surface
Z_filled = pd.DataFrame(Z).interpolate(axis=0).interpolate(axis=1).values

surf = ax.plot_surface(X_grid, Y_grid, Z_filled, cmap=cm.viridis,
                       alpha=0.85, edgecolor="none", antialiased=True)

ax.set_xlabel("Days to Expiration")
ax.set_ylabel("Moneyness (K/S)")
ax.set_zlabel("Implied Volatility (%)")
ax.set_title("SPY Implied Volatility Surface")
ax.view_init(elev=25, azim=-60)

fig.colorbar(surf, ax=ax, shrink=0.5, label="IV (%)")
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap view
fig, ax = plt.subplots(figsize=(10, 8))

im = ax.imshow(Z_filled, cmap="RdYlGn_r", aspect="auto",
               extent=[X.min(), X.max(), Y.min(), Y.max()],
               origin="lower")

ax.axhline(y=1.0, color="white", linestyle="--", linewidth=1, alpha=0.8)
ax.set_xlabel("Days to Expiration")
ax.set_ylabel("Moneyness (K/S)")
ax.set_title("SPY IV Surface - Heatmap")

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Implied Volatility (%)")

plt.tight_layout()
plt.show()

## 5. Delta, Gamma, Theta Across the Surface

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

greeks_to_plot = [
    ("delta", "Delta", "RdBu_r"),
    ("gamma", "Gamma", "YlOrRd"),
    ("theta", "Theta", "RdYlGn"),
]

for ax, (col, title, cmap) in zip(axes, greeks_to_plot):
    pivot = surface.pivot_table(
        index="moneyness_bucket", columns="dte", values=col, aggfunc="mean"
    )
    Zg = pd.DataFrame(pivot.values).interpolate(axis=0).interpolate(axis=1).values

    im = ax.imshow(
        Zg, cmap=cmap, aspect="auto",
        extent=[pivot.columns.min(), pivot.columns.max(),
                pivot.index.min(), pivot.index.max()],
        origin="lower",
    )
    ax.axhline(y=1.0, color="black", linestyle="--", linewidth=0.8, alpha=0.6)
    ax.set_xlabel("DTE")
    ax.set_ylabel("Moneyness")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle("SPY Greeks Surface", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Call delta across strikes for each expiration
fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(selected_exps)))

for i, exp_str in enumerate(selected_exps):
    exp_data = surface[surface["expiration"] == exp_str].sort_values("strike")
    if len(exp_data) == 0:
        continue
    dte = exp_data["dte"].iloc[0]
    ax.plot(exp_data["strike"], exp_data["delta"],
            color=colors[i], linewidth=1.5, label=f"{dte} DTE")

ax.axvline(x=spot, color="gray", linestyle="--", linewidth=1, alpha=0.7)
ax.axhline(y=0.5, color="gray", linestyle=":", linewidth=0.8, alpha=0.5)
ax.set_xlabel("Strike ($)")
ax.set_ylabel("Delta")
ax.set_title("Call Delta by Strike and Expiration")
ax.legend()

plt.tight_layout()
plt.show()

## 6. Time-Decay Analysis: ATM Greeks vs Days to Expiration

In [ ]:
# Near-ATM options (moneyness 0.99 - 1.01)
atm = surface[(surface["moneyness"] >= 0.99) & (surface["moneyness"] <= 1.01)].copy()
atm_avg = atm.groupby("dte").agg({
    "theta": "mean",
    "gamma": "mean",
    "vega": "mean",
    "iv": "mean",
}).reset_index().sort_values("dte")

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0, 0].plot(atm_avg["dte"], atm_avg["theta"], "o-", color="#d62728", markersize=8)
axes[0, 0].set_xlabel("Days to Expiration")
axes[0, 0].set_ylabel("Theta ($/day)")
axes[0, 0].set_title("ATM Theta Decay")
axes[0, 0].invert_xaxis()

axes[0, 1].plot(atm_avg["dte"], atm_avg["gamma"], "o-", color="#2ca02c", markersize=8)
axes[0, 1].set_xlabel("Days to Expiration")
axes[0, 1].set_ylabel("Gamma")
axes[0, 1].set_title("ATM Gamma vs DTE")
axes[0, 1].invert_xaxis()

axes[1, 0].plot(atm_avg["dte"], atm_avg["vega"], "o-", color="#1f77b4", markersize=8)
axes[1, 0].set_xlabel("Days to Expiration")
axes[1, 0].set_ylabel("Vega")
axes[1, 0].set_title("ATM Vega vs DTE")
axes[1, 0].invert_xaxis()

axes[1, 1].plot(atm_avg["dte"], atm_avg["iv"] * 100, "o-", color="#9467bd", markersize=8)
axes[1, 1].set_xlabel("Days to Expiration")
axes[1, 1].set_ylabel("Implied Volatility (%)")
axes[1, 1].set_title("ATM IV Term Structure")

plt.suptitle("SPY ATM Greeks vs Time to Expiration", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Theta/Gamma ratio (the carrying cost of gamma exposure)
atm_avg["theta_gamma_ratio"] = atm_avg["theta"].abs() / atm_avg["gamma"]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(atm_avg["dte"], atm_avg["theta_gamma_ratio"],
       color="#ff7f0e", width=3, edgecolor="white")
ax.set_xlabel("Days to Expiration")
ax.set_ylabel("|Theta| / Gamma")
ax.set_title("Cost of Gamma: |Theta|/Gamma Ratio by DTE")

plt.tight_layout()
plt.show()

---

**Next:** [105 - Real-Time Streaming](105_realtime_streaming.ipynb)